# nb_05_status — pipeline status dashboard

Read-only view of the whole pipeline. Shows **configuration**, the **work queue** (what's done,
in-process, waiting, errored, dead-lettered), **error reasons**, **throughput**, and the live
**Search index** document count. Safe to run anytime — it only reads Delta tables + the Search
count endpoint; it never writes.

All state comes from the Delta tables written by the pipeline (`config`, `file_metadata`,
`ingestion_state`, `ingestion_log`, `skipped_log`) — this notebook is just a lens over them.

## Required permissions
The Fabric lakehouse must be attached (so `spark.table(...)` resolves). To read the live Search
document count it also needs Key Vault **secret get** on `kv_name`/`search_key_secret` (same as
the other notebooks). If the key can't be read, every other panel still works.


## 1. Configuration
The effective `config` table that drives every notebook.


In [ ]:
from pyspark.sql import functions as F
cfg = {r['key']: r['value'] for r in spark.table('config').collect()}
display(spark.table('config').orderBy('key'))


## 1b. Live run progress (`run_progress`)
nb_03 upserts one row per run into `run_progress` as it works (Delta MVCC — this reads live
while a run writes, with no locks). `phase` is `started`/`running`/`done`; `processed` vs
`total_claimed` is how far along the current batch is, with throughput (`rate_per_min`) and
`eta_min`. Re-run this cell to refresh while nb_03 is mid-run.


In [ ]:
try:
    rp = spark.table('run_progress').orderBy(F.col('updated_utc').desc())
    latest = rp.limit(1).collect()
    if latest:
        r = latest[0]
        pct = (100.0 * (r['processed'] or 0) / r['total_claimed']) if r['total_claimed'] else 0.0
        print(f"run {r['run_id']}  phase={r['phase']}")
        print(f"  progress   : {r['processed']}/{r['total_claimed']}  ({pct:.0f}%)")
        print(f"  complete   : {r['complete']}   skipped: {r['skipped']}   error: {r['error']}")
        print(f"  throughput : {r['rate_per_min']}/min   eta: {r['eta_min']} min   elapsed: {r['elapsed_s']}s")
        print(f"  last file  : {r['last_outcome']}  {r['last_file']}")
        print(f"  updated_utc: {r['updated_utc']}")
    else:
        print('run_progress is empty — nb_03 has not run yet.')
    print('--- recent runs ---')
    display(rp.limit(10))
except Exception as e:
    print('run_progress table not found yet (created on first nb_03 run):', e)


## 2. Work queue — status breakdown
Every discovered file has a `process_status`. This is the at-a-glance health of the pipeline.

| status | meaning |
| --- | --- |
| `new` / `changed` / `reingest` | **waiting** to be processed by nb_03 |
| `ingesting` | **in process** right now (or stranded if a run crashed) |
| `complete` | indexed successfully |
| `skipped` | intentionally not indexed (no ACL, unsupported type, empty) |
| `error` | failed, will retry (under `max_retries`) |
| `dead_letter` | failed past `max_retries` — needs attention |
| `deleted` | removed from S3, pending chunk purge |


In [ ]:
WAITING = ['new', 'changed', 'reingest']
counts = {r['process_status']: r['count'] for r in
          spark.table('file_metadata').groupBy('process_status').count().collect()}
total = sum(counts.values())
waiting = sum(counts.get(s, 0) for s in WAITING)
print(f'TOTAL discovered files : {total}')
print(f'  complete            : {counts.get("complete", 0)}')
print(f'  waiting to process  : {waiting}   (new/changed/reingest)')
print(f'  in process (ingesting): {counts.get("ingesting", 0)}')
print(f'  skipped             : {counts.get("skipped", 0)}')
print(f'  error (will retry)  : {counts.get("error", 0)}')
print(f'  dead_letter         : {counts.get("dead_letter", 0)}')
print(f'  deleted (to purge)  : {counts.get("deleted", 0)}')
display(spark.table('file_metadata').groupBy('process_status').count().orderBy('process_status'))


## 3. What still needs processing
Files the next `nb_03` run will pick up (waiting or retryable error, under the retry cap).


In [ ]:
MAX_RETRIES = int(cfg.get('max_retries', '3'))
todo = (spark.table('file_metadata')
        .where((F.col('process_status').isin(WAITING + ['error'])) &
               (F.coalesce(F.col('retry_count'), F.lit(0)) < MAX_RETRIES))
        .select('file_path', 'file_extension', 'process_status', 'status_reason',
                'retry_count', 'status_updated_utc')
        .orderBy('process_status', 'file_path'))
print('files queued for the next nb_03 run:', todo.count())
display(todo)


## 4. In process now
Files currently claimed as `ingesting`. If a file has been here longer than
`ingesting_lease_minutes`, the next nb_03 run reclaims it (crash recovery).


In [ ]:
display(spark.table('file_metadata').where(F.col('process_status') == 'ingesting')
        .select('file_path', 'status_updated_utc', 'retry_count')
        .orderBy('status_updated_utc'))


## 5. Errors & skips — grouped reasons
`skipped_log` captures *why* each file was skipped or failed (the durable error log).


In [ ]:
print('--- skip/error reasons (all runs) ---')
display(spark.table('skipped_log').groupBy('reason').count().orderBy(F.col('count').desc()))
print('--- most recent 25 skip/error rows (with detail) ---')
display(spark.table('skipped_log').orderBy(F.col('ts_utc').desc())
        .select('ts_utc', 'reason', 'file_path', 'detail').limit(25))


## 6. Dead-letter queue
Files that exhausted `max_retries`. These won't be retried automatically — inspect the detail,
fix the root cause, then set their `process_status` back to `reingest` to requeue.


In [ ]:
dl = spark.table('file_metadata').where(F.col('process_status') == 'dead_letter')
print('dead-lettered files:', dl.count())
display(dl.select('file_path', 'status_reason', 'retry_count', 'status_updated_utc'))


## 7. Throughput & volume
From `ingestion_log` (one row per successfully indexed file).


In [ ]:
il = spark.table('ingestion_log')
if il.count() == 0:
    print('no successful ingestions logged yet')
else:
    agg = il.agg(F.count('*').alias('files'), F.sum('chunks').alias('chunks'),
                 F.sum('pages').alias('pages'), F.avg('duration_ms').alias('avg_ms'),
                 F.max('duration_ms').alias('max_ms')).collect()[0]
    print(f'indexed files : {agg["files"]}')
    print(f'total chunks  : {agg["chunks"]}')
    print(f'total pages   : {agg["pages"]}')
    print(f'avg / max per-file time : {agg["avg_ms"]:.0f} ms / {agg["max_ms"]} ms')
    print('--- per-run summary ---')
    display(il.groupBy('run_id').agg(F.count('*').alias('files'),
            F.sum('chunks').alias('chunks'), F.max('ts_utc').alias('last_activity'))
            .orderBy(F.col('last_activity').desc()))


## 8. Live Azure AI Search document count
Chunks actually present in the index right now (via the Search REST count endpoint). Compare this
to `ingestion_state.chunk_count` totals to confirm the index and the state table agree.


In [ ]:
import requests, notebookutils
try:
    VAULT_URL = f"https://{cfg['kv_name']}.vault.azure.net/"
    key = notebookutils.credentials.getSecret(VAULT_URL, cfg['search_key_secret'])
    ep = cfg['search_endpoint'].rstrip('/')
    idx = cfg['search_index_name']
    r = requests.get(f'{ep}/indexes/{idx}/docs/$count?api-version=2024-07-01',
                     headers={'api-key': key}, timeout=(10, 30))
    r.raise_for_status()
    print(f'Search index "{idx}" chunk count : {r.text.strip()}')
except Exception as e:
    print('could not read Search count:', e)

try:
    st = spark.table('ingestion_state').agg(F.count('*').alias('files'),
         F.sum('chunk_count').alias('chunks')).collect()[0]
    print(f'ingestion_state         : {st["files"]} files, {st["chunks"]} chunks expected')
except Exception as e:
    print('no ingestion_state yet:', e)


## 9. Per-folder rollup (optional)
Status by top two path segments — useful to see which areas are fully indexed vs pending.


In [ ]:
seg = F.regexp_replace(F.col('file_path'), '.*/Files/[^/]+/', '')
df = (spark.table('file_metadata')
      .withColumn('area', F.regexp_extract(seg, '^([^/]+/[^/]+)', 1))
      .groupBy('area').pivot('process_status').count().na.fill(0)
      .orderBy('area'))
display(df)
